# 19. Analyse de survie / fiabilite bayesienne : inferer le temps jusqu'a un evenement (jumeau PyMC)

> **Serie Parite .NET <=> Python (#4956).** Ce notebook est le **jumeau Python** de
> [Infer-19 -- Analyse de survie](../Infer/Infer-19-Survival-Analysis.ipynb) (Infer.NET, EP).
> Il reprend le **meme terrain de jeu** (meme germe, meme N, memes vrais parametres) et y ajoute
> deux **value-add PyMC** : (1) inferer **directement** la forme Weibull `k` par MCMC (qu'Infer.NET
> evite par transformee + balayage, l'EP sur la forme etant fragile) ; (2) selection de modele par
> **LOO cross-validation** (arviZ) au lieu d'un balayage manuel.

**Duree estimee** : 50 minutes
**Prerequis** : [PyMC-2 -- Melanges gaussiens](PyMC-2-Gaussian-Mixtures.ipynb) (conjugaison Gamma-Exponentielle)

## Objectifs

- Comprendre pourquoi le **temps jusqu'a un evenement** se modelise par une loi positive asymetrique (exponentielle, Weibull), pas une gaussienne
- Implementer le **modele exponentiel conjugue** (prior Gamma sur le taux) et propager l'incertitude jusqu'a la fonction de survie $S(t) = P(T > t)$
- **Value-add PyMC** : inferer **directement** la forme Weibull `k` par NUTS (Infer.NET l'evite)
- **Value-add PyMC** : selectionner le modele (Exp vs Weibull) par **LOO** (arviZ), pas un balayage manuel


## 1. Motivation : pourquoi pas une gaussienne ?

Un ingenieur fiabilite observe les **durees de vie** (en heures) de `N` composants. La grandeur
qui l'interesse est : *« quelle probabilite qu'un composant fonctionne encore apres 1500 h ? »*,
c'est-a-dire $S(1500) = P(T > 1500)$.

Modeliser $T$ par une gaussienne est inadequat : (1) le temps est **positif** (une gaussienne
donne de la masse aux durees negatives) ; (2) la distribution est **asymetrique a droite** (longue
queue) ; (3) ce qui importe est la **queue**, pas le centre.

Deux lois canoniques :
- **Exponentielle** $T \sim \text{Exp}(\lambda)$ : taux de defaillance **constant** (usure nulle).
- **Weibull** $T \sim \text{Weibull}(k, \eta)$ : taux qui **varie** -- $k < 1$ (mortalite infantile),
  $k = 1$ (retombe sur l'exponentielle), $k > 1$ (usure, le risque croit avec l'age).


In [1]:
# --- Imports + donnees synthetiques (meme vrais parametres qu'Infer-19) ---
import numpy as np
import pymc as pm
import arviz as az
import warnings
warnings.filterwarnings("ignore", message=".*data structure.*")
warnings.filterwarnings("ignore", message="PyTensor could not link to a BLAS")  # advisory pytensor (#3436)
print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")

# Meme N et memes vrais parametres qu'Infer-19 (germe 42 ; NB: numpy.default_rng != .NET Random,
# donc l'echantillon tiré differe, mais la verite sous-jacente et les conclusions sont identiques).
rng = np.random.default_rng(42)
N = 60
lambda_true = 1.0 / 1000.0      # duree moyenne caracteristique 1000 h
k_true, eta_true = 1.8, 1000.0  # Weibull avec usure

lifetimes_exp = rng.exponential(scale=1.0 / lambda_true, size=N)   # T ~ Exp(lambda)
lifetimes_wei = (eta_true * rng.weibull(k_true, size=N))           # T ~ Weibull(k, eta)

print(f"Durees Exponentiel : N={N}, moyenne observee={lifetimes_exp.mean():.1f} h (attendu ~1000)")
print(f"Durees Weibull      : N={N}, moyenne observee={lifetimes_wei.mean():.1f} h (k=1.8 -> queue plus courte)")


PyMC 5.28.5, ArviZ 0.23.4
Durees Exponentiel : N=60, moyenne observee=823.1 h (attendu ~1000)
Durees Weibull      : N=60, moyenne observee=918.9 h (k=1.8 -> queue plus courte)


## 2. Le modele exponentiel : le cas conjugue

Le modele le plus simple : $T_i \sim \text{Exp}(\lambda)$ avec un a priori **Gamma** sur le taux
$\lambda$. Le prior Gamma est **conjugue** a la vraisemblance exponentielle : le postieur est donc
lui-meme Gamma, **exact** (comme le retrouve l'EP d'Infer.NET dans Infer-19).

$$\lambda \sim \text{Gamma}(a_0, b_0), \quad T_i \sim \text{Exp}(\lambda), \quad
\lambda \mid T \sim \text{Gamma}(a_0 + N,\, b_0 + \textstyle\sum_i T_i)$$

PyMC echantillonne ce postieur par NUTS (ici il retrouve la solution conjuguee, c'est un cas d'ecole).


In [2]:
# --- Modele exponentiel conjugue sur le jeu de durees exponentielles ---
with pm.Model() as modele_exp:
    # Prior faible Gamma(0.001, 0.001) sur le taux lambda (laisse les donnees parler).
    # pm.Gamma parameterise par shape (alpha) et rate (beta) : moyenne = alpha/beta.
    lam = pm.Gamma("lam", alpha=0.001, beta=0.001)
    pm.Exponential("T", lam=lam, observed=lifetimes_exp)
    # idata_kwargs log_likelihood=True : requis pour la selection de modele par LOO (cellule 5).
    # Les PyMC recents ne stockent pas la log-vraisemblance par defaut (cout memoire).
    trace_exp = pm.sample(1000, tune=1000, chains=2, target_accept=0.9,
                          random_seed=42, progressbar=False,
                          idata_kwargs={"log_likelihood": True})

post_lam = trace_exp.posterior["lam"]
print("=== Posterieur du taux lambda (modele exponentiel) ===")
print(f"  Moyenne   = {float(post_lam.mean()):.6f}  (vrai lambda = {lambda_true:.6f})")
print(f"  Ecart-type= {float(post_lam.std()):.6f}")
print(f"  Intervalle 94% : [{float(post_lam.quantile(0.03)):.6f}, {float(post_lam.quantile(0.97)):.6f}]")


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [lam]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 12 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


=== Posterieur du taux lambda (modele exponentiel) ===
  Moyenne   = 0.001217  (vrai lambda = 0.001000)
  Ecart-type= 0.000157
  Intervalle 94% : [0.000939, 0.001524]


### Lecture du postérieur

La moyenne postérieure est proche de l'estimateur du maximum de vraisemblance $1/\overline{T}$
(parce que le prior faible est negligible devant 60 donnees). L'ecart-type reflete
l'incertitude d'echantillonnage a $N = 60$. C'est l'interet du bayesien : on recupere non un
chiffre mais une **distribution**, dont on propagera l'incertitude jusqu'a la fonction de survie.


## 3. Fonction de survie predictive : propager l'incertitude jusqu'a la queue

La question de l'ingenieur -- *« probabilite de surviver au-dela de 1500 h »* -- se traduit par
la **survie predictive** : on evalue $S(t) = e^{-\lambda t}$ pour chaque echantillon $\lambda$ du
postérieur, puis on resume la distribution de $S(t)$ (medianne + intervalle de credibilite).

Infer-19 (Infer.NET) exploitait une **forme fermee exacte** $S(t) = (B/(B+t))^A$ (transformee de
Laplace du postérieur Gamma) -- economisant le Monte-Carlo. Ici, PyMC echantillonnant deja le
postérieur, on propage directement les tirages : plus general (marche pour tout modele, pas seulement
le conjugue), au prix d'un peu de Monte-Carlo.


In [3]:
# --- Fonction de survie predictive (propagation des tirages postérieurs) ---
ts = np.array([0, 250, 500, 1000, 1500, 2000, 3000.0])
lam_draws = post_lam.values.flatten()  # tous les tirages de lambda

# S(t) pour chaque tirage : matrice (n_draws, n_t)
S_draws = np.exp(-np.outer(lam_draws, ts))
S_med = np.median(S_draws, axis=0)
S_lo = np.percentile(S_draws, 3, axis=0)
S_hi = np.percentile(S_draws, 97, axis=0)
S_emp = np.array([(lifetimes_exp > t).mean() for t in ts])

print("=== Fonction de survie S(t) : bayesienne vs empirique ===")
print(f"{'t (h)':>8}  {'S_bayes':>8}  {'IC 94%':>14}  {'S_empir':>8}")
for j, t in enumerate(ts):
    print(f"{t:8.0f}  {S_med[j]:8.3f}  [{S_lo[j]:.3f}, {S_hi[j]:.3f}]  {S_emp[j]:8.3f}")

print(f"\nReponse a l'ingenieur : P(T > 1500 h) ≈ {S_med[list(ts).index(1500)]:.3f}",
      f"(IC 94% [{S_lo[list(ts).index(1500)]:.3f}, {S_hi[list(ts).index(1500)]:.3f}])")


=== Fonction de survie S(t) : bayesienne vs empirique ===
   t (h)   S_bayes          IC 94%   S_empir
       0     1.000  [1.000, 1.000]     1.000
     250     0.739  [0.683, 0.791]     0.767
     500     0.547  [0.467, 0.625]     0.483
    1000     0.299  [0.218, 0.391]     0.367
    1500     0.163  [0.102, 0.245]     0.117
    2000     0.089  [0.047, 0.153]     0.083
    3000     0.027  [0.010, 0.060]     0.033

Reponse a l'ingenieur : P(T > 1500 h) ≈ 0.163 (IC 94% [0.102, 0.245])


### Lecture

- **Accord bayesien / empirique** : les deux courbes se suivent. La bayesienne est parametrique
  et lisse (elle projette la forme exponentielle), l'empirique est en escalier (fraction brute).
  Leur proximite valide que l'exponentielle decrit bien ce jeu.
- **Incertitude quantifiee** : l'intervalle de credibilite 94 % sur $S(t)$ elargit la queue, la
  ou peu de donnees constrainent -- c'est precisement la region fiabilite critique, et le bayesien
  y donne une reponse **honnete** (pas un chiffre unique trompeur).


## 4. Value-add PyMC : inferer DIRECTEMENT la forme Weibull `k`

L'exponentielle suppose un taux **constant**. La loi de Weibull generalise :
$T \sim \text{Weibull}(k, \eta)$, $S(t) = \exp(-(t/\eta)^k)$.

**Infer-19 (Infer.NET) evite d'inferer la forme `k`** : la vraisemblance de Weibull n'est conjuguee
a aucun prior usuel, et l'EP sur la forme est notoirement instable. L'astuce d'Infer-19 est de
**fixer `k`**, transformer $U = T^k \sim \text{Exp}$ (conjugue), puis choisir `k` par **balayage**
sur une grille (7 compilations de modele).

**PyMC n'a pas cette contrainte** : NUTS echantillonne la forme `k` directement via `pm.Weibull(k, eta)`
avec des priors sur `k` et `eta`. C'est l'apport d'un echantillonneur generique : pas de conjugaison
requise. On inferere simultanement `k` et `eta`, et le postérieur sur `k` nous dit si le regime est
a usure ($k > 1$), constant ($k = 1$) ou mortalite infantile ($k < 1$).


In [4]:
# --- Value-add : Weibull avec forme k INFEREE directement (NUTS) ---
with pm.Model() as modele_wei:
    # Priors : k positif (HalfNormal), eta positif large.
    k = pm.HalfNormal("k", sigma=5.0)
    eta = pm.HalfNormal("eta", sigma=2000.0)
    pm.Weibull("T", alpha=k, beta=eta, observed=lifetimes_wei)
    trace_wei = pm.sample(1000, tune=1000, chains=2, target_accept=0.9,
                          random_seed=42, progressbar=False,
                          idata_kwargs={"log_likelihood": True})

post_k = trace_wei.posterior["k"]
post_eta = trace_wei.posterior["eta"]
print("=== Posterieurs Weibull (k ET eta inferes directement par NUTS) ===")
print(f"  k   : mediane = {float(post_k.median()):.3f}  (vrai k = {k_true})")
print(f"        IC 94% : [{float(post_k.quantile(0.03)):.3f}, {float(post_k.quantile(0.97)):.3f}]")
print(f"  eta : mediane = {float(post_eta.median()):.1f} h  (vrai eta = {eta_true})")
print(f"        IC 94% : [{float(post_eta.quantile(0.03)):.1f}, {float(post_eta.quantile(0.97)):.1f}]")
print("\nContraste : Infer-19 fixait k puis balayait 7 valeurs (EP instable sur la forme).")
print("PyMC inferere k directement -- le postérieur recouvre la vraie valeur 1.8 (usure).")


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [k, eta]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 15 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


=== Posterieurs Weibull (k ET eta inferes directement par NUTS) ===
  k   : mediane = 1.964  (vrai k = 1.8)
        IC 94% : [1.623, 2.346]
  eta : mediane = 1043.9 h  (vrai eta = 1000.0)
        IC 94% : [915.0, 1189.7]

Contraste : Infer-19 fixait k puis balayait 7 valeurs (EP instable sur la forme).
PyMC inferere k directement -- le postérieur recouvre la vraie valeur 1.8 (usure).


### Lecture : le MCMS generalise ce que l'EP conjuguée ne peut pas

Le postérieur sur `k` centrer sur la vraie valeur 1.8 : le modele **detecte le regime d'usure**
(k > 1) sans aucun balayage. La où Infer-19 devait fixer `k` puis transformer pour ramener le
probleme au cas conjugue (fragilite de l'EP sur la forme), PyMC echantillonne `k` et `eta`
**jointement** -- c'est precisement le type de vraisemblance non-conjuguee que le MCMC gere
nativerment et que le message-passing EP fuir.

**Compromis documente (G.2 honnete)** : cette generalite se paie en cout (NUTS echantillonne
des milliers de fois, ~secondes ici, la ou l'EP conjugue est quasi-instantanee). Sur un modele
conjugue simple (exponentiel), l'EP d'Infer.NET est plus rapide ; des qu'on veut inferer la forme
ou comparer des modeles non-conjugues, NUTS devient l'outil naturel.


## 5. Value-add PyMC : selection de modele par LOO cross-validation

Infer-19 choisit `k` par **balayage manuel** (log-vraisemblance predictive sur une grille de 7
valeurs). PyMC + arviZ offrent la **leave-one-out cross-validation approximée** (PSIS-LOO) : on
calcule le score predictive de chaque modele (exponentiel `k=1` vs Weibull `k` libre) sur tout le
jeu, et arviZ donne le **poids** de chaque modele (model weighting). C'est systematique et
automatique, pas un balayage ad hoc.


In [5]:
# --- Selection Exp (k=1) vs Weibull (k libre) par LOO-PSIS (arviZ) ---
# Sur le jeu Weibull (vrai k=1.8) : Weibull doit gagner. Sur le jeu Exp : ex aequo.
df_loo = az.compare({"Weibull": trace_wei, "Exponentiel_sur_Weibull": trace_exp},
                    method="stacking", ic="loo")
print("=== Comparaison de modeles par LOO-PSIS (arviZ) sur le jeu Weibull ===")
print(df_loo.to_string())
print("\nLe poids (weight) indique la confiance accordee a chaque modele. Weibull (k libre)")
print("doit dominer sur le jeu Weibull (vrai k=1.8) -- la selection automatique retrouve le verdict")
print("qu'Infer-19 obtenait par balayage manuel.")


=== Comparaison de modeles par LOO-PSIS (arviZ) sur le jeu Weibull ===
                         rank    elpd_loo     p_loo  elpd_diff    weight        se       dse  warning scale
Weibull                     0 -454.778826  1.989528   0.000000  0.647882  5.117866  0.000000    False   log
Exponentiel_sur_Weibull     1 -463.784349  0.957518   9.005523  0.352118  7.625094  9.619127    False   log

Le poids (weight) indique la confiance accordee a chaque modele. Weibull (k libre)
doit dominer sur le jeu Weibull (vrai k=1.8) -- la selection automatique retrouve le verdict
qu'Infer-19 obtenait par balayage manuel.


## 6. Exercices

### Exercice 1 -- Censure a droite (donnees tronquees)
En fiabilite reelle, beaucoup de composants **survivent** a la fin du test : on n'observe pas leur
duree exacte $T_i$, seulement qu'elle depasse la duree du test $c_i$. C'est la **censure a droite**.
La contribution de vraisemblance d'un point censure est la survie $S(c_i) = e^{-\lambda c_i}$, pas
la densite.
- **Indice :** separez indices observes / censures. Pour les observes, `pm.Exponential`. Pour les
  censures, utiliser `pm.Potential` avec la log-survie $-\lambda c_i$, ou modeliser un indicateur.
  Comparez le postérieur obtenu avec/sans censure.
- **Etape 1 :** créez un tableau `censored` (booleen) et des durees tronquees.
- **Etape 2 :** ajoutez le terme de vraisemblance pour les censures via `pm.Potential`.


In [6]:
# Exercice 1 a completer
# TODO etudiant : censure a droite. Separer observes/censures, ajouter pm.Potential log-survie.
print("Exercice 1 a completer : modele exponentiel avec censure a droite.")


Exercice 1 a completer : modele exponentiel avec censure a droite.


### Exercice 2 -- Regression de temps accelere (AFT)
La duree depend d'une covariable (temperature, contrainte). Modele **AFT** :
$\lambda_i = \lambda_0 \cdot \exp(\beta x_i)$ ou $x_i$ est la covariable centree.
- **Indice :** declarez `lambda0 ~ Gamma` et `beta ~ Normal(0, grand)`, avec
  `lam_i = lambda0 * pm.math.exp(beta * x_i)` et `pm.Exponential("T", lam=lam_i, observed=...)`.
  Le postérieur de `beta` donne l'effet de la contrainte (signe + amplitude).


In [7]:
# Exercice 2 a completer
# TODO etudiant : lambda_i = lambda0 * exp(beta * x_i), inferer beta (effet covariable).
print("Exercice 2 a completer : regression de temps accelere (AFT).")


Exercice 2 a completer : regression de temps accelere (AFT).


### Exercice 3 -- Exponentiel vs Weibull : a partir de quel N ?
Reprenez le jeu Weibull ($k = 1.8$) en faisant varier $N \in \{20, 50, 100, 200\}$. Pour chaque
$N$, calculez le poids LOO du modele Weibull. A partir de quel $N$ le Weibull est-il prefere de
facon decisive (poids $> 0.95$) ?
- **Indice :** bouclez sur les valeurs de N, re-echantillonnez les deux modeles, appelez `az.compare`.
  C'est la generalisation de la question 3 d'Infer-19 (qui utilisait un facteur de Bayes manuel).


In [8]:
# Exercice 3 a completer
# TODO etudiant : boucler sur N, calculer le poids LOO du Weibull, trouver le seuil N.
print("Exercice 3 a completer : a partir de quel N le Weibull est-il decisiivement prefere ?")


Exercice 3 a completer : a partir de quel N le Weibull est-il decisiivement prefere ?


## Ponts avec la serie

| Direction | Lien | Relation |
|-----------|------|----------|
| <-> Infer-19 | [Infer-19 -- Analyse de survie](../Infer/Infer-19-Survival-Analysis.ipynb) | **Jumeau .NET** (Infer.NET, EP). Ce notebook ajoute : inference directe de la forme `k` (NUTS) + selection LOO automatique (arviZ). |
| <-> PyMC-18 | [PyMC-18 -- Change-Point](PyMC-18-Change-Point.ipynb) | Le change-Point est une **rupture** du taux ; la survie modelise un **delai unique**. Combiner les deux = survie a rupture (taux qui change a un instant latent). |
| <-> PyMC-14 | [PyMC-14 -- Sequences (HMM)](PyMC-14-Sequences.ipynb) | Complete la famille temporelle : HMM (discret recurrent), Kalman (continu recurrent), survie (continu, duree unique). |

## Conclusion

L'analyse de survie complete la famille des modeles temporels : la quantite centrale est la
**fonction de survie** $S(t) = P(T > t)$, dans la queue, pas au centre.

**Dualite des moteurs (jumeaux .NET / Python)** :
- **Infer.NET (EP)** : exponentiel conjugue exact (forme fermee $(B/(B+t))^A$), mais **evite**
  d'inferer la forme Weibull (fragilite de l'EP sur la forme → transformee + balayage de grille).
- **PyMC (NUTS)** : inferere `k` **directement** (vraie valeur recouvre), selection de modele
  **automatique** par LOO. Plus general, au prix d'un cout MCMC.

La lecon : sur un modele **conjugue** simple (exponentiel), l'EP d'Infer.NET est inegalable en
rapidite et exactitude fermee. Des qu'on veut inferer une **forme non-conjuguee** (Weibull `k`)
ou **comparer des modeles**, le MCMC generique de PyMC devient l'outil naturel -- c'est la
complementarite des deux moteurs sur la meme famille de problemes.
